In [1]:
import modules.data as d
import modules.utils as u
from pathlib import Path

#### init ####
dataset_dir = Path('/home/mv18gs/Documents/GitHub/pathway_model/datasets/')
device = u.Devices().auto_set_device(drop=['cuda:4'])

#### data ####
brca = d.TCGA(
    tcga_project = 'BRCA',
    tcga_dir = dataset_dir/'tcga',
    # type_col = 'sample_type',
    subtype_col = 'paper_BRCA_Subtype_PAM50',
    drop = ['Normal', 'Primary Tumor', 'Metastatic'],
    gene_name_path = dataset_dir/'other'/'name2ensg.csv',
    keep_noname = False,
)

kegg = d.KEGG(
    relation_filepath=dataset_dir/'other'/'relation_ohe.csv', 
    counts_data=brca,
)

dataset = d.GraphDataset(brca, kegg, kegg)
_batch = d.get_toy_databatch(dataset, device)

('cuda:4', 'NVIDIA A100-SXM4-80GB', 81043)
('cuda:6', 'NVIDIA A100-SXM4-80GB', 81043)
('cuda:2', 'NVIDIA A100-SXM4-80GB', 81041)
('cuda:3', 'NVIDIA A100-SXM4-80GB', 81041)
('cuda:7', 'NVIDIA A100-SXM4-80GB', 81041)
('cuda:0', 'NVIDIA A100-SXM4-80GB', 78941)
('cuda:5', 'NVIDIA A100-SXM4-80GB', 74623)
('cuda:1', 'NVIDIA A100-SXM4-80GB', 65571)

# #### Device() ####
# device = cuda:6

# #### KEGG() ####
# _orig_kwargs             5                        dict
# relation                 (75939, 19)              DataFrame
# ensg                     4373                     list
# pathway_labels           305                      list
# edge_index               (2, 32464)               Tensor (cuda:6)
# edge_attr                (32464, 16)              Tensor (cuda:6)
# edge_labels              16                       list
# pathway_index            (4373, 305)              Tensor (cuda:6)

# #### TCGA() ####
# _orig_kwargs             9                        dict
# counts_path            

In [2]:
from modules.layers import MultiheadSetPooling
from modules.model import Dims, Encoder, Latent, BaseAutoencoder
from modules.norm import LogCounts, NBVST
from modules.train import Loader
from modules.utils import dict_summary
import torch
import torch.nn as nn

In [3]:
loader = Loader(
    dataset,
    device=device,
    batch_size=128,
)

dims = Dims(
    dataset=dataset,
    embed_dim=64,
    num_heads=4,
    method='set'
)

enc = Encoder(
    dims=dims,
    method='set',

    norm_class=NBVST,
    encoder_class=nn.Linear,
    pooling_class=MultiheadSetPooling,

    hidden_dims=2,
    act_fn=nn.ReLU,
    norm_fn='layer',
)

lat = Latent(
    dims=dims,
    pooling_class=MultiheadSetPooling,

    hidden_dims=2,
    act_fn=nn.ReLU,
    norm_fn='layer',
)

enc.init_with_loader(loader)

out = enc(_batch, need_weights=True)
out = lat(out, need_weights=True)

print(out.keys())

dict_keys(['x', 'x_t', 'layer_outs', 'h_node', 'h_pool', 'z_mu', 'z_logvar', 'z'])


In [10]:
loader = Loader(
    dataset,
    device=device,
    batch_size=128,
)

ae = BaseAutoencoder(
    dataset=dataset,
    embed_dim=64,
    num_heads=4,
    method='set',

    norm_class=NBVST,
    encoder_class=nn.Linear,
    pooling_class=MultiheadSetPooling,
    variational=True,

    hidden_dims=2,
    act_fn=nn.ReLU,
    norm_fn='layer',
)

ae.init_with_loader(loader)

out = ae(_batch, need_weights=True)
# out = ae(_batch, need_weights=False)
if isinstance(out, torch.Tensor):
    print(out.shape)
elif isinstance(out, dict):
    print(dict_summary(out))
else:
    print(out)

# x_t                      (279872, 1)              Tensor (cuda:6)
# layer_outs               4                        dict
# h_node                   (64, 4373, 64)           Tensor (cuda:6)
# h_pool                   (64, 305, 64)            Tensor (cuda:6)
# z_mu                     (64, 64)                 Tensor (cuda:6)
# z_logvar                 (64, 64)                 Tensor (cuda:6)
# z                        (64, 64)                 Tensor (cuda:6)
# x_preds                  (279872, 1)              Tensor (cuda:6)



---

In [11]:
from modules.trainers import ReconstrTrainer
from modules.norm import RawCounts, LogCounts, NBVST

In [13]:
loader = Loader(
    dataset,
    device=device,
    batch_size=128,
)

trainer = ReconstrTrainer(1e-4, norm_class=LogCounts)

ae = BaseAutoencoder(
    dataset=dataset,
    embed_dim=128,
    num_heads=4,
    method='set',

    norm_class=LogCounts,
    encoder_class=nn.Linear,
    pooling_class=MultiheadSetPooling,
    variational=True,

    hidden_dims=2,
    act_fn=nn.ReLU,
    norm_fn='layer',

    norm_kwargs={'libnorm':True, 'znorm':True, 'learnable':True}
)

trainer.run(
    model=ae,
    loader=loader,
    num_epochs=50,
    report_metrics=['loss','rmse','mae','r2'],
    verbose=True
)

  0%|          | 0/50 [00:00<?, ?it/s]

Test	 loss=0.8717    rmse=0.8542    mae=0.5790    r2=0.8129

